# 週次ユニットコミットメント + 送電混雑(簡易DC潮流)

電力会社の「需給運用部」が、翌週(数十〜168時間)の発電ユニット群の起動/停止・出力配分を
決める。各ユニットは特定のバス(送電網のノード)に接続されており、その出力は
**簡易DC潮流(PTDF: Power Transfer Distribution Factor)を通じて全ての送電線の潮流制約に
結合する**。単一ユニットの起動判断が、離れた送電線の混雑を通じて他の全ユニットの経済性に
波及する — これは近似ではなく、ISO/RTOの実務(市場清算・混雑管理)で標準的に使われる
線形結合そのものである。

各制約の業務的意味:

- **起動/停止ロジック・最小連続運転/停止・ランプ**: 火力ユニットは急な出力変更や頻繁な
  起動停止ができない(タービンの熱応力・燃焼安定性)。
- **予備力(スピニングリザーブ)**: 需要急変・ユニット脱落に備え、オンラインユニットの
  最大出力合計は実供給量に安全マージンを乗せた値以上でなければならない。
- **簡易DC潮流(PTDF)**: 各送電線の潮流は「全バスの正味注入電力の線形結合」で決まり、
  熱容量の上下限を超えてはならない。
- **計画外停電(load shedding)バックストップ**: 送電混雑や供給力不足でどうしても需要を
  満たせない場合に、超高コストの計画外停電を許容する(実行可能性の担保)。

この問題の難度は非線形の弱さではなく、**大規模な混合整数計画(MILP)としての組合せ**
(起動停止×PTDFの密結合×週次多期)にある。題材:
`samples/energy_and_microgrid/weekly_uc_ramp.py`。

In [1]:
import sys, pathlib
ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").is_file() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
for sub in ["samples", "samples/energy_and_microgrid"]:
    p = str(ROOT / sub)
    if p not in sys.path:
        sys.path.insert(0, p)

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import HTML, display

def show(fig):
    display(HTML(fig.to_html(include_plotlyjs="cdn", full_html=False,
                             config={"displayModeBar": False})))

import minlpkit as mk
import weekly_uc_ramp as uc
from pyscipopt import Model, quicksum

C = dict(ink="#0b0b0b", ink2="#52514e", muted="#898781", grid="#e1e0d9",
         axis="#c3c2b7", surface="#fcfcfb", s1="#2a78d6", s2="#008300", warn="#c25a00")
FONT = 'system-ui, -apple-system, "Segoe UI", sans-serif'

def base_layout(title, xtitle, ytitle, height=380):
    ax = dict(gridcolor=C["grid"], linecolor=C["axis"],
              tickfont=dict(color=C["muted"], size=11),
              title_font=dict(color=C["ink2"], size=12), zeroline=False)
    return go.Layout(
        title=dict(text=title, font=dict(color=C["ink"], size=15, family=FONT), x=0.01),
        paper_bgcolor=C["surface"], plot_bgcolor=C["surface"],
        font=dict(family=FONT, color=C["ink2"], size=12),
        xaxis=dict(ax, title=xtitle), yaxis=dict(ax, title=ytitle),
        margin=dict(l=60, r=20, t=48, b=48), height=height, hovermode="closest",
        legend=dict(orientation="h", yanchor="bottom", y=1.0, x=1.0, xanchor="right",
                    font=dict(size=11, color=C["ink2"]), bgcolor="rgba(0,0,0,0)"))
print("repo root:", ROOT)

repo root: C:\work_space\mip


## 1. 素朴な定式化

既定scaleはユニット15×バス8×送電線10×48時間。この問題は**線形**(非線形制約ゼロ)で、
難度は起動停止(バイナリ)× PTDFの密結合×多期という組合せの規模そのものにある。

In [2]:
m0 = uc.build_model("default")
print(f"変数 {m0.getNVars()}(うちバイナリ {m0.getNBinVars()})、制約 {m0.getNConss()}")
print("非線形制約:", sum(1 for c in m0.getConss() if c.isNonlinear()), "件(この問題は純粋MILP)")
kinds = {}
for c in m0.getConss():
    k = c.name.rsplit("_", 1)[0] if "_" in c.name else c.name
    kinds[k] = kinds.get(k, 0) + 1
pd.Series(kinds).sort_values(ascending=False).head(10)
del m0

変数 3984(うちバイナリ 2160)、制約 8559


非線形制約: 0 件(この問題は純粋MILP)


`flow_ub`/`flow_lb`(送電線熱容量制約)が最も本数が多い — 送電線×期の数だけあり、
各制約は $\sum_{\text{bus}} \mathrm{PTDF}[l,\text{bus}]\cdot(\text{発電}-\text{需要}+\text{停電})$ という**全バスにまたがる線形結合**を持つ。
これが「単一ユニットの起動判断が離れた送電線を通じて全ユニットへ波及する」密結合の実体。

## 2. 診断

In [3]:
report = mk.analyze(lambda: uc.build_model("default"), name="weekly-uc-ramp",
                    time_limit=30)
print(report.summary())
print("\ngap:", f"{report.metrics.get('gap', 0):.2%}", "/ ノード数:", report.metrics.get("nodes"))

[weekly-uc-ramp] 検出症状 2件:
  [warning] 係数の絶対値レンジが桁違い / Big-M候補あり(presolve後も残存) -> スケーリング、Big-MのIndicator/SOS制約化、係数の再定式化
  [good] 入替可能な変数群(対称性)を検出(参考情報) -> 通常は対応不要(SCIPが自動処理)。usesymmetryを無効化した運用でのみ辞書式除去が有効

gap: 0.02% / ノード数: 1


この規模ではgapはほぼ0%(SCIPのpresolve/heuristicsがルート付近で最適に近い解を即座に
見つける)。`numerical_scale`(presolve後も残る係数スケールの桁違い)だけが発火する —
gapという意味では「難しくない」が、係数レンジの広さそのものは残る。中身を見る。

## 3. 診断の中身を見る — 係数スケールと条件数

`numerical_scale` が指す残存Big-M候補を`static_diag`で直接確認し、線形計画としての
条件数($\kappa(A)$)も測る。

In [4]:
from minlpkit.collectors.static_diag import residual_scale, matrix_condition, scip_basis_condition

m1 = uc.build_model("default")
rs = residual_scale(m1)
print(f"presolve後の係数比 = {rs['ratio']:.3g}(中央値 {rs['median']:.3g})")
bigm_df = pd.DataFrame(rs["bigm"])
bigm_df

presolve後の係数比 = 3.19e+05(中央値 1)


,name,source,magnitude
0,shed_4_12,変数境界,572.512528
1,shed_4_37,変数境界,566.520791
2,shed_4_36,変数境界,562.775092
3,shed_4_10,変数境界,557.631769
4,shed_4_14,変数境界,555.243987
5,shed_4_11,変数境界,549.990071
6,shed_4_38,変数境界,544.687011
7,shed_4_13,変数境界,543.856974
8,shed_4_35,変数境界,542.733676
9,shed_4_34,変数境界,540.115310


In [5]:
fig = go.Figure(layout=base_layout(
    "presolve後に残る大係数(Big-M候補、上位10件)",
    "制約/変数名", "係数の絶対値(対数軸)", height=420))
fig.add_trace(go.Bar(x=bigm_df["name"], y=bigm_df["magnitude"],
    marker_color=[C["warn"] if s == "制約係数" else C["s1"] for s in bigm_df["source"]],
    text=bigm_df["source"], textposition="outside"))
fig.update_yaxes(type="log")
fig.update_xaxes(tickangle=-30)
show(fig)

上位は `reserve_*`(スピニングリザーブ: $\sum \mathrm{pmax}\cdot u \ge \mathrm{served}\cdot(1+\mathrm{reserve})$、係数`pmax`が
数十〜百のオーダー)や `su_cost`(起動費、数百〜1500)まわり。これらは物理量・コストとして
自然な値であり、真の「不必要に緩いBig-M」ではない — `p<=pmax*u`/`p>=pmin*u` の連動制約
(ユニットの出力上下限とon/offのリンク)がBig-M形をしているので、この部分を Indicator化して
効果を確かめる。

## 4. 改善を試す — 出力↔on/offの連動制約を Indicator に置換

`p[i,t] <= pmax[i]*u[i,t]` と `p[i,t] >= pmin[i]*u[i,t]` は「off なら出力0、on なら
`[pmin,pmax]`」という論理をBig-M(`pmax`/`pmin`)で表現したもの。`weekly_uc_ramp.py`には
変種を切り替える引数が無いため、この notebook 内で `build_model` と同じ構築ロジックを
再現しつつ、この2本だけを `addConsIndicator` に置き換えた変種を組み立てる
(`p`の変数境界は`ub=pmax[i]`のまま残るため、off側は `p<=0` の Indicator一発、
on側の下限だけ Indicator で課せば良い)。

In [6]:
def build_indicator(scale="default"):
    '''p<=pmax*u / p>=pmin*u (Big-M) を addConsIndicator に置き換えた変種。'''
    d = uc._data(scale)
    nU, nB, nL, nT = d["nU"], d["nB"], d["nL"], d["nT"]
    edges, PTDF, cap = d["edges"], d["PTDF"], d["cap"]
    pmin, pmax = d["pmin"], d["pmax"]
    a_cost, b_cost, su_cost = d["a_cost"], d["b_cost"], d["su_cost"]
    ramp, mu, md, bus_of = d["ramp"], d["mu"], d["md"], d["bus_of"]
    demand = d["demand"]

    m = Model("Weekly_UC_Ramp_Network_Indicator")
    U, B, L, T = range(nU), range(nB), range(nL), range(nT)
    u, v, w, p, fcv = {}, {}, {}, {}, {}
    for i in U:
        for t in T:
            u[i, t] = m.addVar(vtype="B", name=f"u_{i}_{t}")
            v[i, t] = m.addVar(vtype="B", name=f"v_{i}_{t}")
            w[i, t] = m.addVar(vtype="B", name=f"w_{i}_{t}")
            p[i, t] = m.addVar(vtype="C", lb=0.0, ub=float(pmax[i]), name=f"p_{i}_{t}")
            fcv[i, t] = m.addVar(vtype="C", lb=0.0, name=f"fc_{i}_{t}")
    shed = {(b_, t): m.addVar(vtype="C", lb=0.0, ub=float(demand[b_, t]), name=f"shed_{b_}_{t}")
            for b_ in B for t in T}
    for i in U:
        for t in T:
            # off (u=0) なら出力0。on (u=1) のときだけ下限pminを課す(上限は変数境界ub=pmaxで既に厳密)
            m.addConsIndicator(p[i, t] <= 0, binvar=u[i, t], activeone=False, name=f"ind_off_{i}_{t}")
            m.addConsIndicator(-p[i, t] <= -float(pmin[i]), binvar=u[i, t], activeone=True,
                               name=f"ind_min_{i}_{t}")
            prev = u[i, t - 1] if t > 0 else 0
            m.addCons(v[i, t] - w[i, t] == u[i, t] - prev)
            m.addCons(v[i, t] + w[i, t] <= 1)
            if t > 0:
                m.addCons(p[i, t] - p[i, t - 1] <= float(ramp[i]) * u[i, t - 1] + float(pmin[i]) * v[i, t])
                m.addCons(p[i, t - 1] - p[i, t] <= float(ramp[i]) * u[i, t] + float(pmax[i]) * w[i, t])
            for tau in range(t + 1, min(t + int(mu[i]), nT)):
                m.addCons(u[i, tau] >= v[i, t])
            for tau in range(t + 1, min(t + int(md[i]), nT)):
                m.addCons(u[i, tau] <= 1 - w[i, t])
            m.addCons(fcv[i, t] >= float(a_cost[i]) * u[i, t] + float(b_cost[i]) * p[i, t])
    for t in T:
        m.addCons(quicksum(p[i, t] for i in U) + quicksum(shed[b_, t] for b_ in B)
                  == float(demand[:, t].sum()), name=f"balance_{t}")
        served = float(demand[:, t].sum()) - quicksum(shed[b_, t] for b_ in B)
        m.addCons(quicksum(float(pmax[i]) * u[i, t] for i in U)
                  >= served * (1 + uc.RESERVE), name=f"reserve_{t}")
    gen_at_bus = {(b_, t): quicksum(p[i, t] for i in U if int(bus_of[i]) == b_)
                  for b_ in B for t in T}
    for l in L:
        for t in T:
            net = quicksum(float(PTDF[l, b_]) * (gen_at_bus[b_, t] - float(demand[b_, t]) + shed[b_, t])
                           for b_ in B)
            m.addCons(net <= float(cap[l]), name=f"flow_ub_{l}_{t}")
            m.addCons(net >= -float(cap[l]), name=f"flow_lb_{l}_{t}")
    obj = quicksum(fcv[i, t] + float(su_cost[i]) * v[i, t] for i in U for t in T)
    obj += quicksum(uc.VOLL * shed[b_, t] for b_ in B for t in T)
    m.setObjective(obj, "minimize")
    m.data = dict(u=u, p=p, shed=shed, scale=scale, dims=(nU, nB, nL, nT))
    return m

mb = uc.build_model("small"); mb.hideOutput(); mb.optimize()
mi = build_indicator("small"); mi.hideOutput(); mi.optimize()
print(f"baseline  : obj={mb.getObjVal():.2f}  status={mb.getStatus()}")
print(f"indicator : obj={mi.getObjVal():.2f}  status={mi.getStatus()}")

baseline  : obj=2700675.42  status=optimal
indicator : obj=2700675.42  status=optimal


small scaleで同じ最適値に到達する(浮動小数点誤差の範囲で一致)。既定scaleでは
2節で見た通りgapが既にほぼ0%なので、`compare_variants`の`final_gap`/`nodes`では改善が
見えにくい。ここでは加えて **条件数 $\kappa(A)$** (静的、presolve前のSVD)と
**SCIP基底の条件数**(実際に解いた最適基底、`getCondition()`)も比較する。

In [7]:
df = mk.compare_variants(
    {"baseline (Big-M)": lambda: uc.build_model("default"),
     "indicator":        lambda: build_indicator("default")},
    time_limit=30)
df[["variant", "root_dual", "final_dual", "final_gap", "nodes"]]

,variant,root_dual,final_dual,final_gap,nodes
0,baseline (Big-M),4.773011e+07,4.773011e+07,0.000178,1
1,indicator,4.772984e+07,4.771604e+07,0.002621,1


In [8]:
kappa_base = matrix_condition(uc.build_model("default"))
kappa_ind = matrix_condition(build_indicator("default"))
basis_base = scip_basis_condition(uc.build_model("default"))
basis_ind = scip_basis_condition(build_indicator("default"))
print(f"静的 κ(A)     : baseline={kappa_base['kappa']:.3g}  indicator={kappa_ind['kappa']:.3g}")
print(f"SCIP基底κ     : baseline={basis_base}  indicator={basis_ind}")

静的 κ(A)     : baseline=4.6e+04  indicator=2.76e+07
SCIP基底κ     : baseline=235171.3088610089  indicator=260790.483053453


In [9]:
base = df.loc[df["variant"] == "baseline (Big-M)"].iloc[0]
ind  = df.loc[df["variant"] == "indicator"].iloc[0]
labels = ["baseline (Big-M)", "indicator"]
bar_colors = [C["muted"], C["s1"]]
fig = make_subplots(rows=1, cols=3, horizontal_spacing=0.12,
    subplot_titles=("ルート双対境界", "探索ノード数", "静的 κ(A)(低いほど良い)"))
def add_bars(col, values, fmt):
    fig.add_trace(go.Bar(x=labels, y=values, marker_color=bar_colors,
        text=[fmt(v) for v in values], textposition="outside",
        cliponaxis=False, showlegend=False), row=1, col=col)
add_bars(1, [base["root_dual"], ind["root_dual"]], lambda v: f"{v:.0f}")
add_bars(2, [base["nodes"], ind["nodes"]], lambda v: f"{int(v)}")
add_bars(3, [kappa_base["kappa"], kappa_ind["kappa"]], lambda v: f"{v:.2e}")
fig.update_layout(paper_bgcolor=C["surface"], plot_bgcolor=C["surface"],
    font=dict(family=FONT, color=C["ink2"], size=12), height=360,
    margin=dict(l=40, r=20, t=64, b=40),
    title=dict(text="p<=>u の Indicator化 before / after", x=0.01,
               font=dict(color=C["ink"], size=15)))
fig.update_yaxes(gridcolor=C["grid"], linecolor=C["axis"], zeroline=False)
fig.update_xaxes(linecolor=C["axis"])
show(fig)
print(f"root_dual : {base['root_dual']:.1f} -> {ind['root_dual']:.1f}")
print(f"final_gap : {base['final_gap']:.4%} -> {ind['final_gap']:.4%}")
print(f"nodes     : {int(base['nodes'])} -> {int(ind['nodes'])}")

root_dual : 47730106.7 -> 47729839.6
final_gap : 0.0178% -> 0.2621%
nodes     : 1 -> 1


**正直な検証結果**: `root_dual`/`final_gap`/`nodes`はbaselineから既にほぼ最適(ノード数1、
gap0.02%)であり、Indicator化してもこれらの指標はほとんど動かない(nodesは1のまま、
final_gapはむしろ0.02%->0.26%とわずかに悪化)——改善の余地自体が既にpresolve/heuristicsで
埋められている。

**さらに意外な実測**: 静的 $\kappa(A)$ は baseline=4.6e4 に対し indicator=2.76e7 と**600倍近く悪化**し、
SCIP基底 $\kappa$(getCondition())も 235171 -> 260790 とわずかに悪化した。Indicator制約はSCIP内部で
「制約の有効/無効を切り替えるスラック変数」を介した線形制約として保持されるため、
静的な係数行列の抽出(`extract_coefficients`が拾う線形制約の係数)では**Big-Mを消したはずが
別のスケール差を持ち込む**ことがある——「Big-M排除=常に数値的に健全化する」という
直感に反する実測であり、Indicator化の効果はケースバイケースで実測すべきであることを
裏付ける。**この題材の`numerical_scale`は「実測上の困難」を意味するのではなく、
「発電コスト・出力容量・起動費という業務量が元々桁違いのスケールを持つ」という自然な構造を
指しているに過ぎない**——petroleum_pooling notebookで見た教訓と同様、診断が症状を拾うことと
その症状が実際の求解性能を左右することは別軸である。この問題の本当の難しさは、非線形の
弱さではなく **PTDFによる送電網全体の密結合× 週次多期×数十ユニットの組合せ規模**にあり、
gapを縮めたいなら分解(ベンダーズ/列生成)やチューニングの方が本筋になりうる。

## まとめ

- この問題は**非線形を持たない純粋MILP**でありながら、PTDFによる送電網結合と週次多期の
  規模そのものが難度の源泉になる。既定scaleではSCIPのpresolve/heuristicsが強く、
  30秒でほぼ最適に到達してしまうため、Big-M排除のような緩和強化の効果は実測上ほぼ見えない。
- 実務的には、これは「どのユニットをいつ起動し、送電網の熱容量制約内でどう出力配分するか」
  という需給運用そのものであり、単一バスの需給バランスだけでなく送電線ごとの熱容量制約を
  守る義務がある実務(市場清算・混雑管理)を反映している。
- 教訓: 診断ルールが発火しても、それが「実際に求解を妨げているか」は`compare_variants`で
  必ず確認すべきである。この題材では`numerical_scale`は発火したが、対処してもgap/ノード数は
  動かなかった — 診断は出発点であり、効果測定とセットで初めて意味を持つ。
- さらに意外な教訓: Indicator化は`root_dual`/`final_gap`/`nodes`を悪化させなかった一方、
  静的 $\kappa(A)$・SCIP基底 $\kappa$ は**むしろ悪化**した(4節参照)。「Big-M排除は数値的に常に健全化する」
  という直感は、この題材では反証された — SCIP内部でIndicator制約がスラック変数を介した
  線形制約として保持されるため、係数スケールの見え方自体が変わりうる。

関連: [手法ガイド 3. Big-M排除](../../playbook/03-bigm.md) /
[手法ガイド 8. 条件数・数値健全性](../../playbook/08-condition.md) /
API [`mk.analyze`](../../api/pipeline.md)